# CSCD608 Advanced Computer Vision
## Feature-Based Image Matching & Automatic Panorama Construction

**MPhil/MSc Computer Science | Second Semester Examinations 2025/2026**

This notebook provides an interactive, step-by-step walkthrough of the full pipeline.
It is designed to run in **Google Colab** or a local Jupyter environment.

### Pipeline Overview
```
Input Images → Preprocessing → Feature Detection → Feature Description
→ Descriptor Matching → RANSAC → Homography Estimation
→ Image Warping → Panorama Construction → Quantitative Evaluation
```

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1: Environment Setup
# Run this first — installs required packages
# ═══════════════════════════════════════════════════════════
import sys
import subprocess

# Check if running in Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Running in Google Colab — installing dependencies...')
    subprocess.run(['pip', 'install', '-q', 'opencv-python-headless',
                    'numpy', 'matplotlib', 'pandas', 'scikit-image', 'tqdm'])
    
    # Clone the repo (update URL if needed)
    import os
    REPO_URL = 'https://github.com/mhiskall282/Feature-Based-Image-Matching-and-Automatic-Panorama-Construction-Problem'
    if not os.path.exists('/content/panorama_project'):
        subprocess.run(['git', 'clone', REPO_URL, '/content/panorama_project'])
    os.chdir('/content/panorama_project')
    sys.path.insert(0, '/content/panorama_project')
    print('Repository cloned and path set.')
else:
    # Local — assume we're at the project root
    import os
    project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    sys.path.insert(0, project_root)
    print(f'Local mode. Project root: {project_root}')

print('Setup complete.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2: Imports and version check
# ═══════════════════════════════════════════════════════════
import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Safe in Colab
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

from src.evaluation import get_software_versions
versions = get_software_versions()
print('Library versions:')
for k, v in versions.items():
    print(f'  {k}: {v}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3: Upload or generate images
# ═══════════════════════════════════════════════════════════
from pathlib import Path
import os

# ── Option A: Upload your own images ────────────────────────
# If running in Colab, uncomment and run the upload cell below:

# from google.colab import files
# uploaded = files.upload()  # Select 3+ overlapping images
# for fname, data in uploaded.items():
#     with open(f'data/raw/{fname}', 'wb') as f:
#         f.write(data)

# ── Option B: Use a synthetic scene (no real images needed) ─
# This generates a synthetic scene for demonstration purposes

import cv2
import numpy as np

def make_synthetic_scene(n_images=3, w=640, h=480):
    """Generate overlapping synthetic images for demonstration."""
    # Rich texture: circles + rectangles
    base = np.ones((h, w*2, 3), dtype=np.uint8) * 240
    rng = np.random.RandomState(42)
    for _ in range(80):
        cx, cy = rng.randint(50, w*2-50), rng.randint(50, h-50)
        r  = rng.randint(10, 60)
        color = tuple(int(x) for x in rng.randint(30, 200, 3))
        cv2.circle(base, (cx, cy), r, color, -1)
    for _ in range(40):
        x1, y1 = rng.randint(10, w*2-100), rng.randint(10, h-100)
        x2, y2 = x1+rng.randint(30,120), y1+rng.randint(30,80)
        color = tuple(int(x) for x in rng.randint(30, 200, 3))
        cv2.rectangle(base, (x1,y1), (min(x2,w*2-1),min(y2,h-1)), color, -1)
    
    # Crop overlapping windows
    overlap = w // 3
    step    = w - overlap
    images  = []
    for i in range(n_images):
        x_start = i * step
        crop    = base[:, x_start:x_start+w].copy()
        images.append(crop)
    return images

# Generate synthetic images
data_dir = Path('data/raw')
data_dir.mkdir(parents=True, exist_ok=True)

use_synthetic = True  # Set to False if you have real images in data/raw/

if use_synthetic:
    synth_images = make_synthetic_scene(n_images=3)
    for i, img in enumerate(synth_images):
        cv2.imwrite(str(data_dir / f'scene_img{i+1:02d}.jpg'), img)
    print(f'Generated {len(synth_images)} synthetic scene images → {data_dir}')

# List available images
imgs = sorted(list(data_dir.glob('*.jpg')) + list(data_dir.glob('*.png')))
print(f'Images in data/raw/: {[p.name for p in imgs]}')

if len(imgs) < 2:
    print('WARNING: Need at least 2 images. Add real images to data/raw/ or set use_synthetic=True')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4: Load and display images
# ═══════════════════════════════════════════════════════════
from src.preprocessing import load_image_set, preprocess

img_pairs = load_image_set('data/raw')
images_bgr = [img for _, img in img_pairs]
names      = [p.name for p, _ in img_pairs]

print(f'Loaded {len(images_bgr)} images: {names}')

fig, axes = plt.subplots(1, len(images_bgr), figsize=(7*len(images_bgr), 5))
if len(images_bgr) == 1: axes = [axes]
for ax, img, name in zip(axes, images_bgr, names):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{name}\n{img.shape[1]}×{img.shape[0]} px', fontsize=11)
    ax.axis('off')
fig.suptitle('Input Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/notebook_input.png', dpi=120, bbox_inches='tight')
plt.show()
print('Input images displayed.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5: Feature Detection — SIFT vs ORB
# Pipeline stages: Feature Detection, Feature Description
# REQ-03, REQ-04
# ═══════════════════════════════════════════════════════════
from src.features import detect_and_describe

img1_color, img1_gray = preprocess(images_bgr[0])
img2_color, img2_gray = preprocess(images_bgr[1])

results_table = []
detection_results = {}

for method in ['SIFT', 'ORB']:
    feat1 = detect_and_describe(img1_gray, method)
    feat2 = detect_and_describe(img2_gray, method)
    detection_results[method] = (feat1, feat2)
    results_table.append({
        'Method': method,
        'Keypoints (img1)': feat1['num_kp'],
        'Keypoints (img2)': feat2['num_kp'],
        'Descriptor Dim': feat1['desc_shape'][1] if feat1['desc_shape'] else 'N/A',
        'Descriptor Type': feat1['desc_dtype'],
        'Detection Time (s)': f"{feat1['time_s'] + feat2['time_s']:.3f}",
    })
    print(f'[{method}] img1: {feat1["num_kp"]} kp  img2: {feat2["num_kp"]} kp  '
          f'desc_dim={feat1["desc_shape"]}  time={feat1["time_s"]+feat2["time_s"]:.3f}s')

print()
pd.DataFrame(results_table)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6: Visualize Keypoints
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

for row_idx, method in enumerate(['SIFT', 'ORB']):
    feat1, feat2 = detection_results[method]
    colour = '#2196F3' if method == 'SIFT' else '#FF9800'
    
    for col_idx, (feat, img_gray, img_name) in enumerate([
        (feat1, img1_gray, names[0]),
        (feat2, img2_gray, names[1]),
    ]):
        kp_img = cv2.drawKeypoints(
            img_gray, feat['keypoints'][:500], None,
            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
        )
        axes[row_idx][col_idx].imshow(cv2.cvtColor(kp_img, cv2.COLOR_BGR2RGB))
        axes[row_idx][col_idx].set_title(
            f'{method} — {img_name}\n{feat["num_kp"]} keypoints',
            fontsize=11, color=colour, fontweight='bold'
        )
        axes[row_idx][col_idx].axis('off')

fig.suptitle('SIFT vs ORB — Detected Keypoints (rich circles = scale + orientation)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
Path('outputs').mkdir(exist_ok=True)
plt.savefig('outputs/notebook_keypoints.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7: Descriptor Matching + Initial Correspondences
# Pipeline stage: Descriptor Matching, Initial Correspondences
# REQ-05, REQ-06
# ═══════════════════════════════════════════════════════════
from src.matching import match_descriptors

match_results = {}

for method in ['SIFT', 'ORB']:
    feat1, feat2 = detection_results[method]
    result = match_descriptors(feat1, feat2)
    match_results[method] = result
    print(f'[{method}] raw={result["num_raw_matches"]}  '
          f'good={result["num_good_matches"]}  '
          f'time={result["time_s"]:.3f}s')

# Show raw matches for SIFT
fig, axes = plt.subplots(1, 2, figsize=(22, 6))
for ax, method in zip(axes, ['SIFT', 'ORB']):
    feat1, feat2 = detection_results[method]
    matches = match_results[method]['good_matches'][:150]
    img_m   = cv2.drawMatches(img1_gray, feat1['keypoints'],
                               img2_gray, feat2['keypoints'],
                               matches, None,
                               matchColor=(0, 0, 255),
                               flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    ax.imshow(cv2.cvtColor(img_m, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{method} — Initial Correspondences\n'
                 f'{match_results[method]["num_good_matches"]} good matches', fontsize=11)
    ax.axis('off')
fig.suptitle('Initial Feature Correspondences (Before RANSAC)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/notebook_raw_matches.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8: RANSAC + Homography Estimation
# Pipeline stages: RANSAC, Homography
# REQ-07, REQ-08
# ═══════════════════════════════════════════════════════════
from src.homography import estimate_homography, diagnose_homography

ransac_results = {}

for method in ['SIFT', 'ORB']:
    feat1, feat2 = detection_results[method]
    matches = match_results[method]['good_matches']
    result  = estimate_homography(feat1, feat2, matches)
    ransac_results[method] = result
    print(f'[{method}] inliers={result["num_inliers"]}/{len(matches)}  '
          f'ratio={result["inlier_ratio"]:.1%}  '
          f'reproj_err={result["reprojection_error"]:.2f}px  '
          f'time={result["time_s"]:.3f}s  '
          f'success={result["success"]}')
    if result['H'] is not None:
        print(f'  Homography H:\n{result["H"]}')
        diag = diagnose_homography(result['H'], img1_color.shape, img2_color.shape)
        print(f'  Diagnostics: det={diag["determinant"]:.3f}  degenerate={diag["degenerate"]}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9: Before vs After RANSAC (REQ-11)
# ═══════════════════════════════════════════════════════════
from src.visualization import save_before_after_ransac

for method in ['SIFT', 'ORB']:
    feat1, feat2 = detection_results[method]
    all_m    = match_results[method]['good_matches']
    inlier_m = ransac_results[method]['inlier_matches']
    
    fig, axes = plt.subplots(1, 2, figsize=(22, 6))
    
    before = cv2.drawMatches(img1_gray, feat1['keypoints'],
                              img2_gray, feat2['keypoints'],
                              all_m[:150], None,
                              matchColor=(0,0,255),
                              flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    axes[0].imshow(cv2.cvtColor(before, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f'Before RANSAC: {len(all_m)} matches', fontsize=12)
    axes[0].axis('off')
    
    after = cv2.drawMatches(img1_gray, feat1['keypoints'],
                             img2_gray, feat2['keypoints'],
                             inlier_m, None,
                             matchColor=(0,200,0),
                             flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    ratio = ransac_results[method]['inlier_ratio']
    axes[1].imshow(cv2.cvtColor(after, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f'After RANSAC: {len(inlier_m)} inliers ({ratio:.1%})', fontsize=12)
    axes[1].axis('off')
    
    fig.suptitle(f'{method} — Before vs After RANSAC', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'outputs/notebook_ransac_{method}.png', dpi=130, bbox_inches='tight')
    plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10: Image Warping + Panorama Construction
# Pipeline stages: Warping, Stitching
# REQ-09, REQ-10
# ═══════════════════════════════════════════════════════════
from src.warping import compute_canvas, warp_image
from src.stitching import stitch_pair, stitch_images

for method in ['SIFT', 'ORB']:
    H = ransac_results[method]['H']
    if H is None:
        print(f'[{method}] Homography failed — skipping panorama')
        continue
    
    panorama, info = stitch_pair(img1_color, img2_color, H, alpha_blend=True)
    print(f'[{method}] Panorama: {panorama.shape[1]}×{panorama.shape[0]} px  '
          f'canvas={info["canvas_w"]}×{info["canvas_h"]}')
    
    fig, ax = plt.subplots(figsize=(18, 6))
    ax.imshow(cv2.cvtColor(panorama, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{method} Panorama — {panorama.shape[1]}×{panorama.shape[0]} px',
                 fontsize=13, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'outputs/notebook_panorama_{method}.png', dpi=130, bbox_inches='tight')
    plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 11: Full multi-image panorama
# ═══════════════════════════════════════════════════════════
from src.pipeline import run_multi_image_pipeline

for method in ['SIFT', 'ORB']:
    print(f'\nBuilding full {len(images_bgr)}-image panorama with {method}...')
    panorama, rows = run_multi_image_pipeline(
        images_bgr, names, method,
        out_dir=f'outputs/notebook/{method}',
        experiment='notebook'
    )
    if panorama is not None:
        fig, ax = plt.subplots(figsize=(20, 6))
        ax.imshow(cv2.cvtColor(panorama, cv2.COLOR_BGR2RGB))
        ax.set_title(f'Full {len(images_bgr)}-Image Panorama — {method}\n'
                     f'{panorama.shape[1]}×{panorama.shape[0]} px',
                     fontsize=13, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.savefig(f'outputs/notebook_full_panorama_{method}.png', dpi=130, bbox_inches='tight')
        plt.show()
    else:
        print(f'  [{method}] Panorama failed.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 12: Quantitative Evaluation Summary
# REQ-14: Evaluate and compare the two approaches
# ═══════════════════════════════════════════════════════════
comparison = []
for method in ['SIFT', 'ORB']:
    feat1, feat2 = detection_results[method]
    mr = match_results[method]
    rr = ransac_results[method]
    comparison.append({
        'Method':          method,
        'Descriptor Type': 'float32 128-dim' if method=='SIFT' else 'binary 256-bit',
        'Distance Metric': 'L2 (Euclidean)' if method=='SIFT' else 'Hamming',
        'KP img1':         feat1['num_kp'],
        'KP img2':         feat2['num_kp'],
        'Good Matches':    mr['num_good_matches'],
        'RANSAC Inliers':  rr['num_inliers'],
        'Inlier Ratio':    f"{rr['inlier_ratio']:.2%}",
        'Reproj Err (px)': f"{rr['reprojection_error']:.2f}",
        'Feature Time (s)': f"{feat1['time_s']+feat2['time_s']:.3f}",
        'Match Time (s)':  f"{mr['time_s']:.3f}",
        'RANSAC Time (s)': f"{rr['time_s']:.3f}",
        'H Success':       rr['success'],
    })

df = pd.DataFrame(comparison)
print('SIFT vs ORB Comparison Table')
print('=' * 70)
print(df.to_string(index=False))
df

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 13: Rotation Experiment (mini version)
# REQ-12A: Investigate rotation robustness
# ═══════════════════════════════════════════════════════════
from src.features import detect_and_describe
from src.matching import match_descriptors
from src.homography import estimate_homography

angles  = [0, 15, 30, 45, 60, 90, 120, 180]
h, w    = img1_gray.shape[:2]
exp_rows = []

for method in ['SIFT', 'ORB']:
    for angle in angles:
        cx, cy = w/2, h/2
        M  = cv2.getRotationMatrix2D((cx,cy), float(angle), 1.0)
        cos_a, sin_a = abs(M[0,0]), abs(M[0,1])
        nw = int(h*sin_a + w*cos_a); nh = int(h*cos_a + w*sin_a)
        M[0,2] += nw/2-cx; M[1,2] += nh/2-cy
        rotated = cv2.warpAffine(img1_gray, M, (nw, nh))
        
        f1 = detect_and_describe(img1_gray, method)
        f2 = detect_and_describe(rotated,   method)
        mr = match_descriptors(f1, f2)
        rr = estimate_homography(f1, f2, mr['good_matches'])
        exp_rows.append({'method':method,'angle':angle,
                         'matches':mr['num_good_matches'],
                         'inliers':rr['num_inliers'],
                         'inlier_ratio':rr['inlier_ratio']})

rot_df = pd.DataFrame(exp_rows)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for method, colour in [('SIFT','#2196F3'),('ORB','#FF9800')]:
    sub = rot_df[rot_df['method']==method].sort_values('angle')
    axes[0].plot(sub['angle'], sub['inlier_ratio'], marker='o', color=colour, label=method, lw=2)
    axes[1].plot(sub['angle'], sub['inliers'],      marker='o', color=colour, label=method, lw=2)
for ax, ylabel, title in zip(axes,['Inlier Ratio','RANSAC Inliers'],
                              ['Rotation vs Inlier Ratio','Rotation vs Inlier Count']):
    ax.set_xlabel('Rotation Angle (°)'); ax.set_ylabel(ylabel)
    ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)
fig.suptitle('Rotation Robustness: SIFT vs ORB', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/notebook_rotation.png', dpi=130, bbox_inches='tight')
plt.show()
rot_df

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 14: Save all results
# ═══════════════════════════════════════════════════════════
from pathlib import Path

Path('results').mkdir(exist_ok=True)
rot_df.to_csv('results/notebook_rotation_results.csv', index=False)
df.to_csv('results/notebook_comparison_table.csv', index=False)

print('Saved:')
print('  results/notebook_rotation_results.csv')
print('  results/notebook_comparison_table.csv')
print('  outputs/notebook_*.png')
print('\nAll pipeline stages completed successfully!')
print('Check the outputs/ directory for saved figures.')